# **Programa Especializado en Credit Scoring con Python**
<img src="../../figuras/logo.png" width="200"/>

## 📊 **Sesión 12: Gradient Boosting para Credit Scoring.**

**Docente**: Enzo Infantes Zúñiga  
**Contacto**: <enzo.infantes28@gmail.com>  
**LinkedIn**: [enzo-infantes](https://www.linkedin.com/in/enzo-infantes/)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ptitprince as pt
import seaborn as sns
import math
import sys
import os
import joblib
import warnings
from scipy.stats import randint, uniform

import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import (RandomizedSearchCV, GridSearchCV, StratifiedKFold, cross_val_score, learning_curve)
from sklearn.metrics import (roc_auc_score, roc_curve, auc, confusion_matrix, classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

absolute_path = os.path.dirname(os.path.dirname(os.getcwd()))
data_path = os.path.join(absolute_path, "data", "s10")
src_path = os.path.join(absolute_path, "src", "s12")
model_path = os.path.join(absolute_path, "models", "s12")
figure_path = os.path.join(absolute_path, "figuras", "s12")
sys.path.insert(0, src_path)

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

from evaluation import Evaluation

🎯 **Objetivos de la sesión**

El objetivo de esta sesión es poder entender el funcionamiento de los modelos boosting:

1. Comprender la lógica del Gradient Boosting y cómo supera las limitaciones del Random Forest.
2. Implementar modelos **XGBoost** y **LightGBM** para predecir incumplimiento crediticio.
3. Ajustar hiperparámetros clave y entender su impacto en el desempeño.
4. Evaluar los modelos con métricas propias del credit scoring (Gini, KS, AUC-ROC).
5. Identificar la importancia de variables bajo el enfoque boosting.

**Estructura del Notebook**

```
1. Del Bagging al Boosting: intuición conceptual
2. Carga y preparación de datos
3. XGBoost para Credit Scoring
   3.1 Entrenamiento base
   3.2 Ajuste de hiperparámetros
   3.3 Evaluación del modelo
4. LightGBM para Credit Scoring
   4.1 Entrenamiento base
   4.2 Ajuste de hiperparámetros
   4.3 Evaluación del modelo
5. Importancia de Variables
6. Resumen y buenas prácticas
```



# **1. Boosting: Intuición Conceptual**

El modelo de **Random Forest** se construye con múltiples árboles de manera **independiente y en paralelo** (Bagging), promediando sus predicciones para reducir la varianza.

El **Gradient Boosting** sigue una lógica completamente diferente: construye los árboles de manera **secuencial**, donde cada nuevo árbol corrige los errores del conjunto anterior.

```
┌─────────────────────────────────────────────────────────────────────┐
│                  BAGGING vs BOOSTING                                │
│                                                                     │
│  BAGGING (Random Forest)         BOOSTING (XGBoost/LightGBM)        │
│  ─────────────────────           ────────────────────────────       │
│  Árbol 1 ─┐                      Árbol 1 → residuos                 │
│  Árbol 2 ─┤→ Promedio            Árbol 2 → corrige errores          │
│  Árbol 3 ─┘                      Árbol 3 → corrige errores restantes│
│                                  ...   → Predicción final           │
│  Paralelo, independiente         Secuencial, correctivo             │
└─────────────────────────────────────────────────────────────────────┘
```

| BAGGING (Random Forest) | BOOSTING (XGBoost / LightGBM)    |
| ----------------------- | -------------------------------- |
| Árboles independientes  | Árboles secuenciales             |
| Entrenan en paralelo    | Entrenan uno tras otro           |
| Se promedian resultados | Cada árbol corrige errores       |
| Reduce varianza         | Reduce bias (y también varianza) |


### ¿Cómo funciona el Gradient Boosting?

La idea central es minimizar una función de pérdida de forma iterativa usando **descenso por gradiente**:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

donde:
- $F_{m-1}(x)$ es el modelo acumulado hasta el paso anterior (lo que ya aprendiste).
- $h_m(x)$ es el nuevo árbol que ajusta los **residuos** (para corregir errores, pseudo-residuos).
- $\eta$ es la **tasa de aprendizaje** (*learning rate*), que controla cuánto aporta cada árbol (qué tan grande es el ajuste).
- $F_m(x)$ es el modelo mejorado.

Para clasificación binaria (como el incumplimiento crediticio), la función de pérdida es la **log-loss** (entropía cruzada binaria).

### XGBoost y LightGBM: ¿por qué son tan populares?

| Característica               | XGBoost                      | LightGBM                               |
| ---------------------------- | ---------------------------- | -------------------------------------- |
| 🌳 Estrategia de crecimiento | Nivel por nivel (depth-wise) | Hoja por hoja (leaf-wise)              |
| ⚡ Velocidad                  | Alta                         | Muy alta (ideal para datasets grandes) |
| 🧮 Regularización            | L1 + L2 integradas           | L1 + L2 integradas                     |
| 🏷️ Variables categóricas    | Necesita encoding            | Puede manejarlas directamente          |
| 🏦 Uso en finanzas           | Muy consolidado              | Cada vez más usado                     |


> **Contexto en credit scoring:** Gradient Boosting suele superar a Regresión Logística y Random Forest en métricas de discriminación (Gini, KS), siendo el estándar de facto en competencias de scoring y en producción en muchas instituciones financieras.

# **2. Carga de Datos**

### 2.1 Dataset Simulado de Crédito

In [ ]:
X_train = pd.read_csv(os.path.join(data_path, 'X_train_proc.csv'))
X_test  = pd.read_csv(os.path.join(data_path, 'X_test_proc.csv'))
X_val   = pd.read_csv(os.path.join(data_path, 'X_val_proc.csv'))
y_train = pd.read_csv(os.path.join(data_path, 'y_train.csv'))
y_test  = pd.read_csv(os.path.join(data_path, 'y_test.csv'))
y_val   = pd.read_csv(os.path.join(data_path, 'y_val.csv'))

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"X_val shape: {X_val.shape}")

print(f"y_train defaul rate: {y_train['DEFAULT'].mean()*100:.2f}")
print(f"y_test defaul rate: {y_test['DEFAULT'].mean()*100:.2f}")
print(f"y_val defaul rate: {y_val['DEFAULT'].mean()*100:.2f}")

In [ ]:
X_train.set_index("MES_STR", inplace=True)
X_test.set_index("MES_STR", inplace=True)
X_val.set_index("MES_STR", inplace=True)

# **3. XGBoost para Credit Scoring**

**XGBoost** (eXtreme Gradient Boosting) fue introducido por Chen & Guestrin en 2016 y se convirtió en el algoritmo dominante en competencias de datos. Su popularidad en scoring crediticio radica en:

- **Regularización integrada** (L1 y L2) que controla el overfitting.
- Manejo eficiente de **valores nulos** (los aprende directamente).
- **Poda inteligente** de árboles (*pruning*) basada en ganancia.
- Soporte para **class_weight** y métricas personalizadas.

### **Hiperparámetros clave**

| Hiperparámetro | Descripción | Rango típico | Intuición |
|---|---|---|---|
| `n_estimators` | Número de árboles | 100 – 1000 | (+) árboles -> (+) capacidad aprendizaje / overfitting |
| `learning_rate` (η) | Peso de cada árbol | 0.01 – 0.3 | (-) pasos <-> (+) árboles |
| `max_depth` | Profundidad máxima del árbol | 3 – 8 | (+) profundidad -> (+) overfitting |
| `subsample` | Fracción de filas por árbol | 0.6 – 1.0 | todos los árboles no ven lo mismo |
| `colsample_bytree` | Fracción de columnas por árbol | 0.6 – 1.0 | similar al RF |
| `reg_alpha` (L1) | Regularización Lasso | 0 – 1 | (-) feature irrelevantes |
| `reg_lambda` (L2) | Regularización Ridge | 0 – 5 | (-) feature irrelevantes/overfitting |
| `scale_pos_weight` | Peso para clases desbalanceadas | neg/pos ratio | #negativos / #positivos |

> 💡 **Tradeoff fundamental:** 

`learning_rate` bajo + `n_estimators` alto → mejor generalización, pero mayor tiempo de entrenamiento. En la práctica, se recomienda `learning_rate ≤ 0.1` con early stopping.

### 3.1 Entrenamiento Base con Early Stopping

In [ ]:
scale_pos = (y_train['DEFAULT'] == 0).sum() / (y_train['DEFAULT']  == 1).sum()
print(f"scale_pos_weight sugerido: {scale_pos:.2f} (cada malo es {scale_pos:.2f} veces más importante que cada bueno)")

In [ ]:
# ── XGBoost: modelo base ───────────────────────────────────────────────────────a
xgb_base = xgb.XGBClassifier(
    objective          = 'binary:logistic',
    n_estimators       = 500,
    learning_rate      = 0.05,
    max_depth          = 5,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    scale_pos_weight   = scale_pos,  # manejo del desbalance
    eval_metric        = 'auc',
    early_stopping_rounds = 30,      # detener si no mejora en 30 rondas
    random_state       = SEED,
    verbosity          = 0
)

xgb_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"\nÁrboles óptimos (early stopping): {xgb_base.best_iteration}")
print(f"Mejor AUC en validación:          {xgb_base.best_score:.4f}")

In [ ]:
# ── Curva de aprendizaje ───────────────────────────────────────────────────────
results_xgb = xgb_base.evals_result()

fig, ax = plt.subplots(figsize=(8, 4))
n_rounds = len(results_xgb['validation_0']['auc'])
ax.plot(range(n_rounds), results_xgb['validation_0']['auc'],
        color='#2563EB', lw=2, label='Validación')
ax.axvline(xgb_base.best_iteration, color='red', ls='--', lw=1.5,
           label=f'Early stop = {xgb_base.best_iteration}')
ax.set_xlabel('Número de árboles')
ax.set_ylabel('AUC')
ax.set_title('XGBoost – Curva de Aprendizaje (Validación)')
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Ajuste de Hiperparámetros con Búsqueda Aleatoria

En producción bancaria se utilizan técnicas como **RandomizedSearchCV** o frameworks bayesianos (Optuna). Aquí usamos `RandomizedSearchCV` con validación cruzada estratificada, que es el estándar en la industria.

In [ ]:
# ── Espacio de búsqueda ────────────────────────────────────────────────────────
param_dist_xgb = {
    'n_estimators':     randint(100, 400),
    'max_depth':        randint(3, 7),
    'learning_rate':    uniform(0.01, 0.15),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha':        uniform(0, 1),
    'reg_lambda':       uniform(1, 4),
}

cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        objective='binary:logistic',
        scale_pos_weight=scale_pos,
        eval_metric='auc',
        random_state=SEED,
        verbosity=0
    ),
    param_distributions=param_dist_xgb,
    n_iter=30,
    scoring='roc_auc',
    cv=cv_strat,
    random_state=SEED,
    n_jobs=-1,
    verbose=0
)

xgb_search.fit(X_train, y_train)
xgb_tuned = xgb_search.best_estimator_

print("Mejores hiperparámetros encontrados:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k:<22}: {v:.4f}" if isinstance(v, float) else f"  {k:<22}: {v}")
print(f"\nAUC CV (best): {xgb_search.best_score_:.4f}")

In [ ]:
features = X_train.columns.tolist() 

eval_xgb_test = Evaluation(
    model=xgb_tuned,
    x=X_test,
    y=y_test,
    features=features
)

eval_xgb_test.run_all()

# **4. LightGBM para Credit Scoring**

**LightGBM** fue desarrollado por Microsoft en 2017 y mejora la eficiencia de XGBoost mediante dos innovaciones:

1. **GOSS** (*Gradient-based One-Side Sampling*): conserva las observaciones con gradientes grandes (Las que tienen errores grandes 'gradientes altos' son las más importantes) y muestrea aleatoriamente el resto. Esto acelera el entrenamiento sin perder información relevante.

2. **EFB** (*Exclusive Feature Bundling*): agrupa variables mutuamente excluyentes en un solo feature, reduciendo la dimensionalidad efectiva.

**Diferencia de crecimiento: Depth-wise vs Leaf-wise**

```
XGBoost (depth-wise)         LightGBM (leaf-wise)
────────────────────         ────────────────────
Nivel 1:  [ A ]              Paso 1: [ A ]
           /  \                        /  \
Nivel 2: [B]  [C]            Paso 2: [B]  [C]
         /\   /\                     /\
Nivel 3:[D][E][F][G]         Paso 3:[D][E]    ← solo la hoja con mayor ganancia

→ Más balanceado             → Más profundo donde hay más ganancia
→ Más estable                → Más preciso, potencial overfitting
```

> ⚠️ **Riesgo en credit scoring:** El crecimiento leaf-wise puede causar overfitting en datasets pequeños. Se recomienda controlar con `num_leaves` (parámetro más importante de LightGBM) y `min_child_samples`.

### 4.1 Entrenamiento Base con Early Stopping

In [ ]:
# ── LightGBM: modelo base ──────────────────────────────────────────────────────

lgb_base = lgb.LGBMClassifier(
    objective         = 'binary',
    n_estimators      = 500,
    learning_rate     = 0.05,
    num_leaves        = 31,      # parámetro clave: controla complejidad
    max_depth         = -1,      # sin límite (se controla con num_leaves)
    min_child_samples = 20,      # mínimo de muestras en cada hoja
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    class_weight      = 'balanced',
    random_state      = SEED,
    verbose           = -1
)

callbacks = [
    lgb.early_stopping(stopping_rounds=30, verbose=False),
    lgb.log_evaluation(period=-1)
]

lgb_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=callbacks
)

print(f"Árboles óptimos (early stopping): {lgb_base.best_iteration_}")
print(f"Mejor AUC en validación:          {lgb_base.best_score_['valid_0']['auc']:.4f}")

### 4.2 Ajuste de Hiperparámetros

In [ ]:
# ── Búsqueda aleatoria para LightGBM ──────────────────────────────────────────
param_dist_lgb = {
    'n_estimators':      randint(100, 400),
    'num_leaves':        randint(15, 63),
    'learning_rate':     uniform(0.01, 0.15),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
    'min_child_samples': randint(10, 50),
    'reg_alpha':         uniform(0, 1),
    'reg_lambda':        uniform(0, 3),
}

cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

lgb_search = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(
        objective='binary',
        class_weight='balanced',
        random_state=SEED,
        verbose=-1
    ),
    param_distributions=param_dist_lgb,
    n_iter=30,
    scoring='roc_auc',
    cv=cv_strat,
    random_state=SEED,
    n_jobs=-1,
    verbose=0
)

lgb_search.fit(X_train, y_train)
lgb_tuned = lgb_search.best_estimator_

print("Mejores hiperparámetros encontrados:")
for k, v in lgb_search.best_params_.items():
    print(f"  {k:<22}: {v:.4f}" if isinstance(v, float) else f"  {k:<22}: {v}")
print(f"\nAUC CV (best): {lgb_search.best_score_:.4f}")

In [ ]:
eval_lgb_val = Evaluation(
    model=lgb_tuned,
    x=X_test,
    y=y_test,
    features=features
)

eval_lgb_val.run_all()

In [ ]:
joblib.dump(xgb_tuned, os.path.join(model_path, 'modelo_xgb.pkl'))
joblib.dump(lgb_tuned, os.path.join(model_path, 'modelo_lgb.pkl'))

# **5. Importancia de Variables**

Los modelos de Gradient Boosting calculan la importancia de cada variable según tres criterios:

| Tipo | Descripción | Ventaja |
|---|---|---|
| **Weight** | Frecuencia de uso en los splits | Simple, pero sesgado hacia variables con muchos valores |
| **Gain** | Ganancia media por split | Más representativo del impacto predictivo |
| **Cover** | Cobertura media de muestras en los splits | Mide cuántas observaciones afecta |

> En credit scoring se recomienda usar **Gain** como métrica principal de importancia. Para interpretabilidad profunda, se utilizará SHAP en la Sesión 14.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, {'XGBoost': xgb_tuned, 'LightGBM': lgb_tuned}.items()):
    
    if hasattr(model, 'get_booster'):
        scores = model.get_booster().get_score(importance_type='weight')
        imp = np.array([scores.get(f, 0) for f in features])
    else:
        imp = model.booster_.feature_importance(importance_type='split')

    idx = np.argsort(imp)[::-1][:10]
    colors = ['#2563EB'] * 3 + ['#93C5FD'] * 7

    ax.barh(np.array(features)[idx][::-1], imp[idx][::-1], color=colors)
    ax.set_title(f'Top 10 – {name}', fontweight='bold')
    ax.set_xlabel('Weight / Split')

plt.suptitle('Importancia de Variables: Gradient Boosting', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

# **7. Resumen y Buenas Prácticas**

**Lo que aprendimos en esta sesión**

| Tema | Puntos clave |
|---|---|
| **Gradient Boosting** | Construcción secuencial corrigiendo residuos con descenso por gradiente |
| **XGBoost** | Depth-wise, regularización L1+L2, manejo nativo de nulls, `scale_pos_weight` |
| **LightGBM** | Leaf-wise, más rápido en datasets grandes, clave: `num_leaves` |
| **Early Stopping** | Evita overfitting: detiene el entrenamiento cuando la métrica de validación deja de mejorar |
| **Ajuste de hiperparámetros** | RandomizedSearchCV con CV estratificada es el estándar en producción |
| **Importancia de variables** | Preferir Gain; SHAP para interpretación profunda (Sesión 14) |

**Buenas prácticas en entornos bancarios**

1. **Desbalance de clases:** Siempre usar `scale_pos_weight` (XGBoost) o `class_weight='balanced'` (LightGBM). No depender solo del umbral de decisión.

2. **Early stopping con conjunto de validación:** Nunca usar el test set para early stopping. Usar un conjunto de validación separado.

3. **`num_leaves` en LightGBM:** Regla práctica: `num_leaves < 2^max_depth`. Un valor de 31 es un buen punto de partida.

4. **Validación temporal:** En datos crediticios reales, separar train/test cronológicamente (no aleatoriamente) para simular el uso real del modelo.